In [ ]:
from tqdm import tqdm
import pandas as pd
import ollama
import json
import time
import ast
import os

In [2]:
df_split_fair = pd.read_excel(os.path.join('../Data/PreProcessed', '06_Datasets.xlsx'))
df_clear = df_split_fair[df_split_fair['is_encrypted'] == 0]
df_clear['Columns'] = df_clear['Columns'].apply(ast.literal_eval)
colunas_para_classificar = df_clear['Columns'].explode().dropna().unique().tolist()

In [3]:
MODELO_PRINCIPAL = 'gemma3:12b'
tamanho_lote = 1
dicionario_final = {}

colunas_para_classificar = df_clear['Columns'].explode().dropna().unique().tolist()

print(f"Iniciando o Pipeline Final para {len(colunas_para_classificar)} variáveis apenas com {MODELO_PRINCIPAL}...")

Iniciando o Pipeline Final para 789 variáveis apenas com gemma3:12b...


---

### **| Fase 1 ) -** Prompts

---

In [4]:
prompt_classificacao = """
Role: You are a Senior Financial Data Architect. 
Task: Classify variables according to STRICT mapping rules.
Context: This is for a banking credit scoring system.

RULES FOR MAPPING (PRIORITY ORDER):
1. SOCIOECONOMIC: Home ownership, number of children/dependents, household size, length of employment, employment status, occupation, family income, income, wealth, parents’ income, social class, education. (STRICT: Employment duration and salary go here, NEVER in Institutional).
2. DEMOGRAPHIC: Age, gender, ethnicity, marital status, family life cycle. (STRICT: NO geographical locations here. NO account age here).
3. VALUES, ATTITUDES and BEHAVIORAL: Socio-motivation/others, attitudes toward debt, parent facilitation, time horizon, perceived financial wellbeing, attitudes toward positive financial behavior, religious practices, consumer behavior, level of expenditure (spending pattern), attitudes toward money/money beliefs and behaviors, attitudes toward risk-taking, attitudes toward credit, compulsive buying, delay of gratification, financial knowledge, loan purpose/intent.
4. INSTITUTIONAL and FINANCIAL: Number of debts, length of relationship with the bank, number of bank accounts, debt to income ratio, total financial assets, payment pattern, credit limit, existing credit commitments, credit score, number of credit cards, credit history in the past, loan amount, taking debt advice, loan duration, account balance. (STRICT: Only banking, credit, and debt metrics. NO demographic or income data).
5. PERSONALITY: Self-control, emotional stability/neuroticism, intelligence (IQ score), locus of control, self-efficacy, agreeableness, optimism, self-esteem, extraversion, conscientiousness, openness to experience, impulsiveness.
6. SITUATIONAL: ONLY adverse life events or life-altering events. (STRICT: NEVER classify dates, system terms, or housing here).
7. EDUCATIONAL: Field of study, GPA (Grade Point Average), year at school. (NOTE: General education level goes to SOCIOECONOMIC).
8. MACROECONOMIC: General economic indicators such as inflation (CPI) or economic growth (GDP).
9. HEALTH-RELATED: Physical health status and mental health indicators.
10. ALTERNATIVE: Social network visiting patterns, posting patterns, usage of photo for posts, major things in people’s lives, qualifications of people within individuals’ network, social network prestige, membership score, retweet behavior, emoticon utilization, posting time, number of followers, friends, the slope of daily calls sent, SMS sending pattern, number of messages, disclosure of social media profile, AND ALL geographical locations (City, State, Zipcode), telemetry, device info, and IP addresses.
11. UNCLASSIFIED: System IDs, Keys, hashes, meaningless strings, generic technical flags, variables that lack semantic meaning, and specific system timestamps/calendar dates (e.g., CreationDate, DefaultDate).

EXAMPLES OF CORRECT MAPPING:
- 'housing_type' -> 'SOCIOECONOMIC' (Not Situational!)
- 'city_of_residence' -> 'ALTERNATIVE' (Not Demographic!)
- 'loan_term_months' -> 'INSTITUTIONAL and FINANCIAL'
- 'months_employed' -> 'SOCIOECONOMIC' (Not Institutional!)
- 'ListingKey' -> 'UNCLASSIFIED'

INPUT VARIABLES: {lote}

OUTPUT: Return ONLY a valid JSON object where keys are variable names and values are the categories.
"""

In [5]:
prompt_reflexao = """
Role: You are a Zero-Tolerance Quality Auditor for Data Engineering.
Task: Fix common logical errors in the following JSON classification.

AUDIT CHECKLIST (FIX IF WRONG):
1. Is any geographical location, IP address, or device info in 'DEMOGRAPHIC'? -> MOVE TO 'ALTERNATIVE'.
2. Is 'housing', 'property', 'income', 'salary', or 'employment' in 'INSTITUTIONAL and FINANCIAL'? -> MOVE TO 'SOCIOECONOMIC'.
3. Are raw system dates, timestamps, IDs, or Keys (e.g., 'DateOfBirth', 'ListingKey', 'LoanID') in ANY category other than 'UNCLASSIFIED'? -> MOVE TO 'UNCLASSIFIED'.
4. Is 'SITUATIONAL' being used for anything other than major life tragedies? -> IF SO, REASSIGN IT.
5. Is loan 'purpose', 'intent', or 'motive' in 'INSTITUTIONAL and FINANCIAL'? -> MOVE TO 'VALUES, ATTITUDES and BEHAVIORAL'.
6. Is 'age', 'gender', or 'marital status' in 'SOCIOECONOMIC'? -> MOVE TO 'DEMOGRAPHIC'.

Return ONLY the corrected JSON object. No explanations.
JSON TO AUDIT: {json_original}
"""

---

### **| Fase 2 ) -** Execução

---

In [6]:
for i in tqdm(range(0, len(colunas_para_classificar), tamanho_lote), desc=f"Processando lotes ({MODELO_PRINCIPAL})"):
    lote_atual = colunas_para_classificar[i:i + tamanho_lote]
    
    try:
        prompt_1 = prompt_classificacao.format(lote=lote_atual)
        resp1 = ollama.generate(model=MODELO_PRINCIPAL, prompt=prompt_1, format='json', options={'temperature': 0.0})
        
        prompt_2 = prompt_reflexao.format(json_original=resp1['response'])
        resp2 = ollama.generate(model=MODELO_PRINCIPAL, prompt=prompt_2, format='json', options={'temperature': 0.0})
        
        dicionario_final.update(json.loads(resp2['response']))
        
    except Exception as e:
        print(f"\nErro no lote que inicia em {i}: {e}")

df_macro_categorias = pd.DataFrame(list(dicionario_final.items()), columns=['Col', 'Macro_Categoria'])
df_flattened = df_clear[['id', 'Columns']].explode('Columns').rename(columns={'Columns': 'Col'})

df_final_mapeado = pd.merge(
    df_flattened, 
    df_macro_categorias, 
    on='Col', 
    how='left'
)

print("\nProcessamento e mapeamento de IDs finalizados com sucesso!")
df_final_mapeado.head(10)

Processando lotes (gemma3:12b):   0%|          | 0/789 [00:00<?, ?it/s]

Processando lotes (gemma3:12b): 100%|██████████| 789/789 [57:56<00:00,  4.41s/it] 


Processamento e mapeamento de IDs finalizados com sucesso!


,id,Col,Macro_Categoria
0,CRED-002,status,NaN
1,CRED-002,duration,UNCLASSIFIED
2,CRED-002,credit_history,SOCIOECONOMIC
3,CRED-002,purpose,"VALUES, ATTITUDES and BEHAVIORAL"
4,CRED-002,amount,UNCLASSIFIED
5,CRED-002,savings,SOCIOECONOMIC
6,CRED-002,employment_duration,DEMOGRAPHIC
7,CRED-002,installment_rate,SOCIOECONOMIC
8,CRED-002,personal_status_sex,DEMOGRAPHIC
9,CRED-002,other_debtors,SOCIOECONOMIC
